In [1]:
!pip install -q -U \
    bitsandbytes \
    transformers \
    peft \
    trl \
    datasets \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.8 MB/s eta 0:00:00:00:01


In [2]:
import torch
import transformers
import peft
import trl
import datasets
import bitsandbytes as bnb

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("CUDA available:", torch.cuda.is_available())

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Datasets:", datasets.__version__)
print("BitsAndBytes:", bnb.__version__)

GPU: Tesla T4
CUDA available: True
Transformers: 5.17.0
PEFT: 0.21.0
TRL: 1.14.0
Datasets: 5.0.1
BitsAndBytes: 0.50.2


In [3]:
from datasets import load_dataset

dataset = load_dataset("xlangai/spider")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [4]:
print(dataset["train"][0])

{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']}


In [5]:
from huggingface_hub import hf_hub_download
import zipfile
from pathlib import Path

zip_path = hf_hub_download(
    repo_id="HAL-9001/spider-databases",
    filename="spider_data.zip",
    repo_type="dataset",
)

EXTRACT_DIR = Path("/kaggle/working/spider_assets")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)

spider_data.zip:   0%|          | 0.00/206M [00:00<?, ?B/s]

Extracted to: /kaggle/working/spider_assets


In [6]:
files = list(EXTRACT_DIR.rglob("tables.json"))
print("tables.json:", files)

db_files = list(EXTRACT_DIR.rglob("*.sqlite"))
print("SQLite databases:", len(db_files))

print("First DB:", db_files[0] if db_files else None)

tables.json: [PosixPath('/kaggle/working/spider_assets/spider_data/tables.json')]
SQLite databases: 744
First DB: /kaggle/working/spider_assets/spider_data/test_database/scientist_1/scientist_1.sqlite


In [7]:
import json

TABLES_PATH = "/kaggle/working/spider_assets/spider_data/tables.json"

with open(TABLES_PATH, "r", encoding="utf-8") as f:
    tables_data = json.load(f)

schema_by_db = {
    item["db_id"]: item
    for item in tables_data
}

db_id = dataset["train"][0]["db_id"]

schema = schema_by_db[db_id]

print("Database:", db_id)
print("\nTables and columns:")

table_names = schema["table_names_original"]
column_names = schema["column_names_original"]

for table_id, table_name in enumerate(table_names):
    columns = [
        column_name
        for col_table_id, column_name in column_names
        if col_table_id == table_id
    ]

    print(f"{table_name}({', '.join(columns)})")


print("\nForeign keys:")

for source_idx, target_idx in schema["foreign_keys"]:
    source_table_id, source_column = column_names[source_idx]
    target_table_id, target_column = column_names[target_idx]

    source_table = table_names[source_table_id]
    target_table = table_names[target_table_id]

    print(
        f"{source_table}.{source_column} "
        f"-> {target_table}.{target_column}"
    )

Database: department_management

Tables and columns:
department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
head(head_ID, name, born_state, age)
management(department_ID, head_ID, temporary_acting)

Foreign keys:
management.head_ID -> head.head_ID
management.department_ID -> department.Department_ID


In [8]:
def build_schema_text(db_id):
    schema = schema_by_db[db_id]

    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]

    lines = []

    # Tables + columns
    for table_id, table_name in enumerate(table_names):
        columns = [
            column_name
            for col_table_id, column_name in column_names
            if col_table_id == table_id
        ]

        lines.append(
            f"{table_name}({', '.join(columns)})"
        )

    # Foreign keys
    if schema["foreign_keys"]:
        lines.append("")
        lines.append("Foreign keys:")

        for source_idx, target_idx in schema["foreign_keys"]:
            source_table_id, source_column = column_names[source_idx]
            target_table_id, target_column = column_names[target_idx]

            source_table = table_names[source_table_id]
            target_table = table_names[target_table_id]

            lines.append(
                f"{source_table}.{source_column} "
                f"-> {target_table}.{target_column}"
            )

    return "\n".join(lines)

In [9]:
example = dataset["train"][0]

schema_text = build_schema_text(example["db_id"])

messages = [
    {
        "role": "system",
        "content": (
            "You are a text-to-SQL assistant. "
            "Generate only a valid SQL query using the provided schema. "
            "Do not invent tables or columns."
        )
    },
    {
        "role": "user",
        "content": (
            f"Schema:\n{schema_text}\n\n"
            f"Question:\n{example['question']}"
        )
    },
    {
        "role": "assistant",
        "content": example["query"]
    }
]

for message in messages:
    print(f"\n{message['role'].upper()}:")
    print(message["content"])


SYSTEM:
You are a text-to-SQL assistant. Generate only a valid SQL query using the provided schema. Do not invent tables or columns.

USER:
Schema:
department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
head(head_ID, name, born_state, age)
management(department_ID, head_ID, temporary_acting)

Foreign keys:
management.head_ID -> head.head_ID
management.department_ID -> department.Department_ID

Question:
How many heads of the departments are older than 56 ?

ASSISTANT:
SELECT count(*) FROM head WHERE age  >  56


In [10]:
SYSTEM_PROMPT = (
    "You are a text-to-SQL assistant. "
    "Generate only a valid SQL query using the provided schema. "
    "Do not invent tables or columns."
)


def format_example(example):
    schema_text = build_schema_text(example["db_id"])

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                f"Schema:\n{schema_text}\n\n"
                f"Question:\n{example['question']}"
            ),
        },
        {
            "role": "assistant",
            "content": example["query"],
        },
    ]

    return {"messages": messages}

In [11]:
formatted_dataset = dataset.map(format_example)

print(formatted_dataset)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks', 'messages'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks', 'messages'],
        num_rows: 1034
    })
})


In [12]:
example = formatted_dataset["train"][0]

print("DB:", example["db_id"])
print("Question:", example["question"])
print("Target SQL:", example["query"])

print("\nMessages:")
for message in example["messages"]:
    print(f"\n{message['role'].upper()}:")
    print(message["content"])

DB: department_management
Question: How many heads of the departments are older than 56 ?
Target SQL: SELECT count(*) FROM head WHERE age  >  56

Messages:

SYSTEM:
You are a text-to-SQL assistant. Generate only a valid SQL query using the provided schema. Do not invent tables or columns.

USER:
Schema:
department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
head(head_ID, name, born_state, age)
management(department_ID, head_ID, temporary_acting)

Foreign keys:
management.head_ID -> head.head_ID
management.department_ID -> department.Department_ID

Question:
How many heads of the departments are older than 56 ?

ASSISTANT:
SELECT count(*) FROM head WHERE age  >  56


In [13]:
train_db_ids = set(formatted_dataset["train"]["db_id"])
val_db_ids = set(formatted_dataset["validation"]["db_id"])

missing_schema_train = [
    db_id for db_id in train_db_ids
    if db_id not in schema_by_db
]

missing_schema_val = [
    db_id for db_id in val_db_ids
    if db_id not in schema_by_db
]

empty_train = sum(
    1
    for row in formatted_dataset["train"]
    if not row["question"].strip() or not row["query"].strip()
)

empty_val = sum(
    1
    for row in formatted_dataset["validation"]
    if not row["question"].strip() or not row["query"].strip()
)

db_overlap = train_db_ids & val_db_ids

print("Train databases:", len(train_db_ids))
print("Validation databases:", len(val_db_ids))

print("Missing train schemas:", len(missing_schema_train))
print("Missing validation schemas:", len(missing_schema_val))

print("Empty train examples:", empty_train)
print("Empty validation examples:", empty_val)

print("Train/validation DB overlap:", len(db_overlap))
print("Overlap:", db_overlap)

Train databases: 140
Validation databases: 20
Missing train schemas: 0
Missing validation schemas: 0
Empty train examples: 0
Empty validation examples: 0
Train/validation DB overlap: 0
Overlap: set()


In [14]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [15]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA: True
GPU: Tesla T4


In [16]:
print("tokenizer" in globals())
print("formatted_dataset" in globals())

True
True


In [17]:
import torch

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map={"": 0},
)

print("Model loaded successfully in 4-bit")
print("Loaded in 4-bit:", model.is_loaded_in_4bit)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully in 4-bit
Loaded in 4-bit: True


In [18]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for k-bit training")

Model prepared for k-bit training


In [19]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [20]:
def to_prompt_completion(example):
    return {
        "prompt": example["messages"][:2],
        "completion": example["messages"][2:],
    }


train_sft = formatted_dataset["train"].map(
    to_prompt_completion,
    remove_columns=formatted_dataset["train"].column_names,
)

val_sft = formatted_dataset["validation"].map(
    to_prompt_completion,
    remove_columns=formatted_dataset["validation"].column_names,
)

print(train_sft)
print(val_sft)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 7000
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1034
})


In [21]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen-sql-qlora",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,

    max_length=1024,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    completion_only_loss=True,
    packing=False,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=10,
    save_total_limit=1,

    report_to="none",
)

In [22]:
from trl import SFTTrainer

model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1034 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1034 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1034 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [23]:
print("Final train examples:", len(trainer.train_dataset))
print("Final validation examples:", len(trainer.eval_dataset))

Final train examples: 6918
Final validation examples: 1034


In [24]:
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainer.model.zero_grad(set_to_none=True)

In [25]:
from collections import Counter

print(
    Counter(
        str(p.dtype)
        for p in trainer.model.parameters()
        if p.requires_grad
    )
)

Counter({'torch.float32': 392})


In [26]:
train_result = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.147845,0.504011,0.104987,1821514.000000,0.936742


In [27]:
SAVE_PATH = "/kaggle/working/qwen-text-to-sql-qlora"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved at:", SAVE_PATH)

Model saved at: /kaggle/working/qwen-text-to-sql-qlora


In [28]:
import torch

model.eval()
model.config.use_cache = True

example = formatted_dataset["validation"][0]

prompt_messages = example["messages"][:2]
gold_sql = example["messages"][2]["content"]

prompt_text = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

predicted_sql = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("Question:")
print(example["question"])

print("\nGold SQL:")
print(gold_sql)

print("\nPredicted SQL:")
print(predicted_sql)

Question:
How many singers do we have?

Gold SQL:
SELECT count(*) FROM singer

Predicted SQL:
SELECT count(*) FROM singer


In [29]:
import re

def normalize_sql(sql):
    sql = sql.strip().lower()
    sql = re.sub(r"\s+", " ", sql)
    sql = sql.rstrip(";")
    return sql

In [30]:
from tqdm.auto import tqdm
import torch

model.eval()
model.config.use_cache = True

validation_data = formatted_dataset["validation"]

batch_size = 4
predictions = []
gold_queries = []

tokenizer.padding_side = "left"

for start in tqdm(
    range(0, len(validation_data), batch_size),
    desc="Evaluating"
):
    batch = validation_data[
        start:start + batch_size
    ]

    prompts = []

    for messages in batch["messages"]:
        prompt_text = tokenizer.apply_chat_template(
            messages[:2],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[:, inputs["input_ids"].shape[1]:]

    decoded = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
    )

    predictions.extend([x.strip() for x in decoded])

    gold_queries.extend(
        [messages[2]["content"] for messages in batch["messages"]]
    )

Evaluating:   0%|          | 0/259 [00:00<?, ?it/s]

In [31]:
correct = 0

for pred, gold in zip(predictions, gold_queries):
    if normalize_sql(pred) == normalize_sql(gold):
        correct += 1

accuracy = correct / len(gold_queries)

print("Correct:", correct)
print("Total:", len(gold_queries))
print(f"Normalized Exact Match Accuracy: {accuracy * 100:.2f}%")

Correct: 482
Total: 1034
Normalized Exact Match Accuracy: 46.62%


In [32]:
import sqlite3
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm

DB_ROOT = Path(
    "/kaggle/working/spider_assets/spider_data/database"
)

def execute_query(db_path, query):
    try:
        conn = sqlite3.connect(
            f"file:{db_path}?mode=ro",
            uri=True
        )

        conn.text_factory = lambda b: b.decode(
            "utf-8",
            errors="replace"
        )

        cursor = conn.cursor()
        cursor.execute(query)
        rows = cursor.fetchall()

        conn.close()
        return rows, None

    except Exception as e:
        return None, str(e)


def same_execution_result(pred_rows, gold_rows, gold_sql):
    if pred_rows is None:
        return False

    # If ORDER BY matters, preserve order
    if "order by" in gold_sql.lower():
        return pred_rows == gold_rows

    # Otherwise ignore row order
    return Counter(pred_rows) == Counter(gold_rows)

In [33]:
execution_correct = 0
execution_errors = 0

validation_original = formatted_dataset["validation"]

for i in tqdm(range(len(predictions)), desc="Execution evaluation"):

    db_id = validation_original[i]["db_id"]

    db_path = (
        DB_ROOT /
        db_id /
        f"{db_id}.sqlite"
    )

    pred_sql = predictions[i]
    gold_sql = gold_queries[i]

    pred_rows, pred_error = execute_query(
        db_path,
        pred_sql
    )

    gold_rows, gold_error = execute_query(
        db_path,
        gold_sql
    )

    if pred_error is not None:
        execution_errors += 1
        continue

    if gold_error is not None:
        continue

    if same_execution_result(
        pred_rows,
        gold_rows,
        gold_sql
    ):
        execution_correct += 1


execution_accuracy = (
    execution_correct / len(predictions)
)

print("Execution Correct:", execution_correct)
print("Total:", len(predictions))
print("Prediction Execution Errors:", execution_errors)

print(
    f"Execution Accuracy: "
    f"{execution_accuracy * 100:.2f}%"
)

Execution evaluation:   0%|          | 0/1034 [00:00<?, ?it/s]

Execution Correct: 781
Total: 1034
Prediction Execution Errors: 47
Execution Accuracy: 75.53%


In [34]:
failure_cases = []

for i in range(len(predictions)):
    db_id = formatted_dataset["validation"][i]["db_id"]
    question = formatted_dataset["validation"][i]["question"]

    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"

    pred_sql = predictions[i]
    gold_sql = gold_queries[i]

    pred_rows, pred_error = execute_query(db_path, pred_sql)
    gold_rows, gold_error = execute_query(db_path, gold_sql)

    if pred_error is not None:
        failure_cases.append({
            "type": "execution_error",
            "db_id": db_id,
            "question": question,
            "gold_sql": gold_sql,
            "predicted_sql": pred_sql,
            "error": pred_error,
        })

    elif not same_execution_result(pred_rows, gold_rows, gold_sql):
        failure_cases.append({
            "type": "wrong_result",
            "db_id": db_id,
            "question": question,
            "gold_sql": gold_sql,
            "predicted_sql": pred_sql,
            "error": None,
        })

print("Total failures:", len(failure_cases))
print("Execution errors:", sum(x["type"] == "execution_error" for x in failure_cases))
print("Wrong-result queries:", sum(x["type"] == "wrong_result" for x in failure_cases))

Total failures: 253
Execution errors: 47
Wrong-result queries: 206


In [35]:
from collections import Counter

error_categories = Counter()

for case in failure_cases:
    if case["type"] != "execution_error":
        continue

    error = case["error"].lower()

    if "no such column" in error:
        error_categories["invalid_column"] += 1

    elif "no such table" in error:
        error_categories["invalid_table"] += 1

    elif "ambiguous column" in error:
        error_categories["ambiguous_column"] += 1

    elif "syntax error" in error:
        error_categories["syntax_error"] += 1

    else:
        error_categories["other"] += 1

print(error_categories)

Counter({'invalid_column': 41, 'invalid_table': 4, 'syntax_error': 1, 'other': 1})


In [36]:
execution_correct = 0
execution_errors = 0

for i in range(len(predictions)):
    db_id = formatted_dataset["validation"][i]["db_id"]

    db_path = (
        DB_ROOT /
        db_id /
        f"{db_id}.sqlite"
    )

    pred_sql = predictions[i]
    gold_sql = gold_queries[i]

    pred_rows, pred_error = execute_query(db_path, pred_sql)
    gold_rows, gold_error = execute_query(db_path, gold_sql)

    if pred_error is not None:
        execution_errors += 1
        continue

    if gold_error is not None:
        continue

    if same_execution_result(
        pred_rows,
        gold_rows,
        gold_sql
    ):
        execution_correct += 1

execution_accuracy = execution_correct / len(predictions)

print("Execution Correct:", execution_correct)
print("Total:", len(predictions))
print("Prediction Execution Errors:", execution_errors)
print(f"Execution Accuracy: {execution_accuracy * 100:.2f}%")

Execution Correct: 781
Total: 1034
Prediction Execution Errors: 47
Execution Accuracy: 75.53%


In [37]:
correct = sum(
    normalize_sql(pred) == normalize_sql(gold)
    for pred, gold in zip(predictions, gold_queries)
)

accuracy = correct / len(gold_queries)

print("Correct:", correct)
print("Total:", len(gold_queries))
print(f"Normalized Exact Match Accuracy: {accuracy * 100:.2f}%")

Correct: 482
Total: 1034
Normalized Exact Match Accuracy: 46.62%


In [38]:
from huggingface_hub import notebook_login

notebook_login()

In [40]:
from huggingface_hub import HfApi, create_repo

api = HfApi()

username = api.whoami()["name"]

REPO_NAME = "qwen2.5-coder-7b-text-to-sql-qlora"
REPO_ID = f"{username}/{REPO_NAME}"

create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

print("Repository created:")
print(REPO_ID)

Repository created:
mohd-maaz/qwen2.5-coder-7b-text-to-sql-qlora


In [41]:
from huggingface_hub import HfApi

SAVE_PATH = "/kaggle/working/qwen-text-to-sql-qlora"

# Save trained LoRA adapter + tokenizer
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Upload to Hugging Face
api = HfApi()

api.upload_folder(
    folder_path=SAVE_PATH,
    repo_id=REPO_ID,
    repo_type="model",
)

print("Upload complete!")
print(REPO_ID)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete!
mohd-maaz/qwen2.5-coder-7b-text-to-sql-qlora
